# 00_build_dataset

- Load clean speech utterances from specified subset of LibriSpeech train-clean-100 talkers
    - resample to 16 kHz
    - train: 4 hrs; val & test ~0.5 hrs
- Partition talkers into train/val (251 - 10 = 231 speakers) and test (20 fully held-out speakers, balanced by sex)
- Std-normalize each utterance, sample a fractional clipping threshold α ~ Uniform(0.05, 0.95) applied relative to the - utterance peak
- Chunk train and val utterances into non-overlapping 1024-sample windows/blocks/chunks (interchangeably used)
    - store (clipped, clean) chunk pairs as pre-generated dataset on SSD
- Reserve test utterances as complete files from held-out speakers
    - chunking and overlap-add reconstruction occur at inference time for test data, as it is ordered

In [1]:
# Imports

import os
import random
from pathlib import Path
import math
import json

import numpy as np
import torch
import torchaudio
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
%matplotlib widget

import sys
sys.path.insert(0, "..")
from config import *

In [2]:
# Constants

# 00_data/config.py loads relevant constants

# nb-specific constants
GET_NOISE_FLOOR = False
N_SILENCE_PROBE = 500

In [3]:
# Load librispeech dataset

dataset = torchaudio.datasets.LIBRISPEECH(
    root=LIBRISPEECH_ROOT,
    url="train-clean-100",
    download=True
)

In [4]:
# Subject metadata

tlkr_path = LIBRISPEECH_ROOT / "LibriSpeech" / "SPEAKERS.TXT"
tlkr_df = pd.read_csv(
    tlkr_path,
    sep="|",
    skiprows=11,
    header=0,
    names=["tlkr_id", "sex", "subset", "minutes", "name"],
    skipinitialspace=True,
    on_bad_lines='skip'
)

# Strip whitespace, filter to train-clean-100
tlkr_df["sex"] = tlkr_df["sex"].str.strip()
tlkr_df["subset"] = tlkr_df["subset"].str.strip()
tlkr_df = tlkr_df[tlkr_df["subset"] == "train-clean-100"]

# Manually add speaker 60 (M, train-clean-100) who was dropped due to pipe in name
tlkr_df = pd.concat([
    tlkr_df,
    pd.DataFrame([{"tlkr_id": 60, "sex": "M", "subset": "train-clean-100", "minutes": None, "name": None}])
], ignore_index=True)

print(tlkr_df["sex"].value_counts())
print(f"Total talkers: {len(tlkr_df)}")
tlkr_df.tail(10)

sex
M    126
F    125
Name: count, dtype: int64
Total talkers: 251


/var/folders/bq/csqgvczj6jndm459wzs6n4kw0000gn/T/ipykernel_14518/4116116645.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  tlkr_df = pd.concat([


,tlkr_id,sex,subset,minutes,name
241,8580,M,train-clean-100,19.82,Gary Dana
242,8609,M,train-clean-100,25.10,noblesavage
243,8629,M,train-clean-100,25.13,Shivansh Dhar
244,8630,M,train-clean-100,23.50,Eduardo
245,8747,M,train-clean-100,23.49,DeanOBuchanan
246,8770,M,train-clean-100,25.13,Paul Simonin
247,8797,M,train-clean-100,22.76,Sean Grabosky
248,8838,M,train-clean-100,25.06,Kevin Owens
249,8975,F,train-clean-100,25.11,Daisy Flaim
250,60,M,train-clean-100,NaN,None


In [5]:
# Build test pool: 20 talkers fully held out, 10 m 10 f

random.seed(42)

male_tlkrs = tlkr_df[tlkr_df["sex"] == "M"]["tlkr_id"].tolist()
female_tlkrs = tlkr_df[tlkr_df["sex"] == "F"]["tlkr_id"].tolist()

test_tlkr_m = random.sample(male_tlkrs, N_TEST_TLKR // 2)
test_tlkr_f = random.sample(female_tlkrs, N_TEST_TLKR // 2)
test_tlkrs = set(test_tlkr_m + test_tlkr_f)

print(f"Test talkers M: {sorted(test_tlkr_m)}")
print(f"Test talkers F: {sorted(test_tlkr_f)}")
print(f"Total test talkers: {len(test_tlkrs)}")

Test talkers M: [118, 374, 405, 446, 1081, 1355, 1723, 5463, 6019, 6563]
Test talkers F: [40, 83, 200, 730, 1069, 2691, 4051, 4297, 6064, 7800]
Total test talkers: 20


In [6]:
# Get indices of N shortest utterances per test talker

manifest_path = TEST_OUT / "test_manifest.json"

if manifest_path.exists():
    print("Test set already exists, skipping.")
else:
    # Collect utterance durations for test talkers
    test_tlkr_utts = {tlkr: [] for tlkr in test_tlkrs}
    
    for i, (wav, sr, transcript, tlkr, chap, utt) in enumerate(dataset):
        if tlkr in test_tlkrs:
            duration = wav.shape[1] / sr
            test_tlkr_utts[tlkr].append((duration, i))
        if i % 1000 == 0:
            print(f"Current index: {i}")
    
    # Select N_TEST_UTT_PER_TLKR shortest utterances per talker
    test_utt_indices = []
    for tlkr, utts in test_tlkr_utts.items():
        shortest = sorted(utts, key=lambda x: x[0])[:N_TEST_UTT_PER_TLKR]
        test_utt_indices.extend([i for _, i in shortest])
    
    print(f"Total test utterances: {len(test_utt_indices)}")

Test set already exists, skipping.


In [7]:
# Store test utterances: individual files due to variable length


if manifest_path.exists():
    print("Test set already exists, skipping.")
else:
    TEST_OUT.mkdir(parents=True, exist_ok=True)
    manifest = []

    for file_idx, dataset_idx in enumerate(test_utt_indices):
        wav, sr, transcript, tlkr, chap, utt = dataset[dataset_idx]
        
        if sr != FS:
            wav = torchaudio.functional.resample(wav, sr, FS)
        
        std = wav.std().clamp(min=1e-8)
        wav_norm = wav / std
        
        alpha = random.uniform(ALPHA_MIN, ALPHA_MAX)
        
        clean_path = TEST_OUT / f"test_{file_idx:03d}_clean.pt"
        torch.save(wav_norm, clean_path)
        
        manifest.append({
            "file_idx": file_idx,
            "dataset_idx": dataset_idx,
            "tlkr_id": tlkr,
            "alpha": round(alpha, 6),
            "num_samples": wav_norm.shape[1]
        })

    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    print(f"Saved {len(manifest)} test utterances to {TEST_OUT}")

Test set already exists, skipping.


In [8]:
# Find block-wise noise floor power threshold

if GET_NOISE_FLOOR:
    random.seed(0)
    probe_indices = random.sample(range(len(dataset)), N_SILENCE_PROBE)
    
    block_powers = []
    
    for dataset_idx in probe_indices:
        wav, sr, _, _, _, _ = dataset[dataset_idx]
        
        if sr != FS:
            wav = torchaudio.functional.resample(wav, sr, FS)
        
        std = wav.std().clamp(min=1e-8)
        wav_norm = wav / std
        
        # Non-overlapping blocks, discard tail
        n_blocks = wav_norm.shape[1] // BS
        if n_blocks == 0:
            continue
        blocks = wav_norm[:, :n_blocks * BS].unfold(1, BS, BS)  # (1, n_blocks, BS)
        
        # Avg power per block (mean of squared samples)
        power = blocks.pow(2).mean(dim=2).squeeze(0)  # (n_blocks,)
        block_powers.extend(power.tolist())
    
    block_powers = torch.tensor(block_powers)
    
    fig = px.histogram(
        x=block_powers.numpy(),
        nbins=500,
        log_y=True,
        labels={"x": "Avg block power"},
        title=f"Block power distribution ({N_SILENCE_PROBE} utterances, {len(block_powers)} blocks)"
    )
    fig.show()

In [9]:
# Train/val dataset generation


# Guard
train_path = TRAIN_OUT / "train_blocks.pt"
train_manifest_path = TRAIN_OUT / "train_manifest.json"
val_path = VAL_OUT / "val_blocks.pt"
val_manifest_path = VAL_OUT / "val_manifest.json"

if train_manifest_path.exists() and val_manifest_path.exists():
    print("Train/val sets already exist, skipping.")
else:
    TRAIN_OUT.mkdir(parents=True, exist_ok=True)
    VAL_OUT.mkdir(parents=True, exist_ok=True)

    # Step 1: group non-test dataset indices by talker
    tlkr_to_indices = {}
    for i, (_, _, _, tlkr, _, _) in enumerate(dataset):
        if tlkr not in test_tlkrs:
            if tlkr not in tlkr_to_indices:
                tlkr_to_indices[tlkr] = []
            tlkr_to_indices[tlkr].append(i)

    n_train_tlkrs = len(tlkr_to_indices)
    print(f"Non-test talkers: {n_train_tlkrs}")

    # Step 2: blocks per talker
    total_blocks_needed = (TRAIN_HR + VAL_HR) * 3600 * FS / BS
    blocks_per_tlkr = math.ceil(total_blocks_needed / n_train_tlkrs)
    print(f"Blocks per talker: {blocks_per_tlkr}")

    # Steps 3-4: build global pool
    global_pool = []  # list of (block_tensor, tlkr_id)

    # Set reproducible random seed
    random.seed(42)

    for tlkr, indices in tlkr_to_indices.items():
        tlkr_pool = []

        for dataset_idx in indices:
            wav, sr, _, _, _, _ = dataset[dataset_idx]

            if sr != FS:
                wav = torchaudio.functional.resample(wav, sr, FS)

            std = wav.std().clamp(min=1e-8)
            wav_norm = wav / std

            n_blocks = wav_norm.shape[1] // BS
            if n_blocks == 0:
                continue

            blocks = wav_norm[:, :n_blocks * BS].unfold(1, BS, BS)
            # blocks shape: (1, n_blocks, BS) -> (n_blocks, 1, BS)
            blocks = blocks.squeeze(0).unsqueeze(1)

            for b in range(blocks.shape[0]):
                block = blocks[b]  # (1, BS)
                power = block.pow(2).mean().item()
                if power >= SILENCE_THRESH:
                    tlkr_pool.append((block, tlkr))

        # Per-talker shuffle
        random.shuffle(tlkr_pool)

        # Sample blocks_per_tlkr blocks
        sampled = tlkr_pool[:blocks_per_tlkr]
        global_pool.extend(sampled)

        print(f"Talker {tlkr}: {len(tlkr_pool)} valid blocks, sampled {len(sampled)}")

    # Step 4: global shuffle
    random.shuffle(global_pool)
    print(f"Global pool size: {len(global_pool)}")

    # Step 5: split
    n_train = math.floor(TRAIN_HR / (TRAIN_HR + VAL_HR) * len(global_pool))
    train_pool = global_pool[:n_train]
    val_pool = global_pool[n_train:]
    print(f"Train blocks: {len(train_pool)}, Val blocks: {len(val_pool)}")

    # Step 6: save train and val
    for pool, blocks_path, manifest_path in [
        (train_pool, train_path, train_manifest_path),
        (val_pool, val_path, val_manifest_path)
    ]:
        # Stack into single tensor (N, 1, BS)
        tensor = torch.stack([b for b, _ in pool])
        torch.save(tensor, blocks_path)

        # Sample alpha per block and save manifest
        manifest = []
        for block_idx, (_, tlkr) in enumerate(pool):
            alpha = random.uniform(ALPHA_MIN, ALPHA_MAX)
            manifest.append({
                "block_idx": block_idx,
                "tlkr_id": tlkr,
                "alpha": round(alpha, 6)
            })

        with open(manifest_path, "w") as f:
            json.dump(manifest, f, indent=2)

        print(f"Saved {len(pool)} blocks to {blocks_path}")

Train/val sets already exist, skipping.
